# 01 — Data Understanding (REVISI)

Lanjutan skripsi, direvisi untuk rencana baru:
- Target biner: `0=Baik`, `1=Perlu Pemeliharaan` (Kurang Baik + Rusak Berat)
- Model utama: XGBoost + SMOTE-NC + TreeSHAP
- Fokus baru di notebook ini: cek kategorikal untuk SMOTE-NC, cek skala numerik (harga mendominasi jarak SMOTE), dan cek korelasi umur–depresiasi–harga.

Jalankan di env `ml-inventaris` (Python 3.11). File: `../data/raw/inventaris_raw.csv` — taruh CSV asli di situ dulu.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.options.display.float_format = '{:.2f}'.format
RAW = Path('../data/raw/inventaris_raw.csv')
print('file ada:', RAW.exists(), '| path:', RAW.resolve())

file ada: True | path: D:\Magister\Semester 1\Pembelajaran Mesin\ml-inventaris\data\raw\inventaris_raw.csv


In [2]:
df = pd.read_csv(RAW)
print('shape:', df.shape)
display(df.head())
df.info()

shape: (2322, 9)


,triwulan,nama_ruangan,nama_barang,bahan,kategori,tahun_pembelian,jumlah,harga,kondisi
0,Q1,Adenium,Lemari Sepatu Dua Pintu,Kayu,Furnitur,2014,1,2400000.00,Baik
1,Q1,Adenium,Printer Canon 237,Besi,Elektronik,2014,1,1350000.00,Baik
2,Q1,Adenium,Kursi Roda Spirit BM,Besi,Pelengkap,2014,1,1149500.00,Baik
3,Q1,Adenium,Lemari Sliding Kaca,Besi,Furnitur,2015,2,6490000.00,Baik
4,Q1,Adenium,Meja Nurse Station,Besi,Furnitur,2015,1,5875000.00,Baik


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2322 entries, 0 to 2321
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   triwulan         2322 non-null   object 
 1   nama_ruangan     2322 non-null   object 
 2   nama_barang      2322 non-null   object 
 3   bahan            2322 non-null   object 
 4   kategori         2322 non-null   object 
 5   tahun_pembelian  2322 non-null   int64  
 6   jumlah           2322 non-null   int64  
 7   harga            1029 non-null   float64
 8   kondisi          2322 non-null   object 
dtypes: float64(1), int64(2), object(6)
memory usage: 163.4+ KB


In [3]:
print('MISSING VALUE (jumlah & %)')
miss = pd.DataFrame({'jumlah': df.isnull().sum(), 'persen': (df.isnull().mean()*100).round(2)})
display(miss[miss['jumlah']>0])
print('\nHarga kosong:', int(df['harga'].isnull().sum()), 'dari', len(df), '= 55.7% — ini yang ditangani di notebook 02')

MISSING VALUE (jumlah & %)


,jumlah,persen
harga,1293,55.68



Harga kosong: 1293 dari 2322 = 55.7% — ini yang ditangani di notebook 02


In [4]:
print('NILAI UNIK KATEGORIKAL (untuk SMOTE-NC: 2 fitur kategorikal)')
for c in ['bahan','kategori','kondisi','nama_ruangan','triwulan']:
    print(f"\n{c}: {df[c].nunique()} nilai -> {sorted(df[c].dropna().unique().tolist())}")

NILAI UNIK KATEGORIKAL (untuk SMOTE-NC: 2 fitur kategorikal)

bahan: 5 nilai -> ['Besi', 'Kaca', 'Kain', 'Kayu', 'Plastik']

kategori: 3 nilai -> ['Elektronik', 'Furnitur', 'Pelengkap']

kondisi: 3 nilai -> ['Baik', 'Kurang Baik', 'Rusak Berat']

nama_ruangan: 12 nilai -> ['Adenium', 'Anturium', 'Aster', 'Dahlia', 'Gardena', 'Lavender', 'Mawar', 'Melati A', 'Melati B', 'Sakura', 'Seruni', 'Tulip']

triwulan: 4 nilai -> ['Q1', 'Q2', 'Q3', 'Q4']


In [5]:
print('STATISTIK DESKRIPTIF NUMERIK')
display(df[['tahun_pembelian','jumlah','harga']].describe().T)
print('\nCatatan skala: harga mean ~12 jt, max 398 jt — vs umur 1-15. Tanpa scaling, jarak kNN di SMOTE/SMOTE-NC akan didominasi harga. Ini alasan RobustScaler dipakai di notebook 03.')

STATISTIK DESKRIPTIF NUMERIK


,count,mean,std,min,25%,50%,75%,max
tahun_pembelian,2322.00,2018.91,3.62,2011.00,2016.00,2019.00,2022.00,2025.00
jumlah,2322.00,3.66,13.47,1.00,1.00,1.00,3.00,240.00
harga,1029.00,12257307.24,40358503.33,10000.00,1350000.00,3220000.00,6615000.00,398284500.00



Catatan skala: harga mean ~12 jt, max 398 jt — vs umur 1-15. Tanpa scaling, jarak kNN di SMOTE/SMOTE-NC akan didominasi harga. Ini alasan RobustScaler dipakai di notebook 03.


In [6]:
print('DISTRIBUSI KELAS ASLI (3 kelas)')
print(df['kondisi'].value_counts())
print(df['kondisi'].value_counts(normalize=True).round(4)*100)
print('\nDISTRIBUSI KELAS BINER (rencana baru)')
df['kondisi_biner'] = df['kondisi'].map({'Baik': 0, 'Kurang Baik': 1, 'Rusak Berat': 1})
print(df['kondisi_biner'].value_counts())
n_baik = int((df['kondisi_biner']==0).sum()); n_perlu = int((df['kondisi_biner']==1).sum())
n_rusak = int((df['kondisi']=='Rusak Berat').sum())
print(f'Rasio Baik : Perlu Pemeliharaan = {n_baik} : {n_perlu} = {n_baik/n_perlu:.2f} : 1  (data mentah, sebelum cleaning)')
print(f'Bandingkan Rusak Berat sendirian: {n_baik} : {n_rusak} = {n_baik/n_rusak:.1f} : 1 — jauh lebih timpang')
print('Setelah cleaning di notebook 02 (2.301 baris): 1889 : 412 = 4.59 : 1')

DISTRIBUSI KELAS ASLI (3 kelas)
kondisi
Baik           1909
Kurang Baik     327
Rusak Berat      86
Name: count, dtype: int64
kondisi
Baik          82.21
Kurang Baik   14.08
Rusak Berat    3.70
Name: proportion, dtype: float64

DISTRIBUSI KELAS BINER (rencana baru)
kondisi_biner
0    1909
1     413
Name: count, dtype: int64
Rasio Baik : Perlu Pemeliharaan = 1909 : 413 = 4.62 : 1  (data mentah, sebelum cleaning)
Bandingkan Rusak Berat sendirian: 1909 : 86 = 22.2 : 1 — jauh lebih timpang
Setelah cleaning di notebook 02 (2.301 baris): 1889 : 412 = 4.59 : 1


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x='kondisi', ax=axes[0]); axes[0].set_title('3 kelas (skripsi)')
sns.countplot(data=df, x='kondisi_biner', ax=axes[1]); axes[1].set_title('2 kelas biner (rencana baru: 0=Baik, 1=Perlu)')
plt.tight_layout(); plt.show()

<Figure size 1200x400 with 2 Axes>

In [8]:
print('CEK KORELASI AWAL (umur vs harga vs depresiasi nanti)')
print('Umur dihitung di notebook 02 sebagai REFERENCE_YEAR - tahun_pembelian.')
print('Depresiasi adalah fungsi deterministik dari umur + harga_per_unit + kategori,')
print('jadi ketiganya akan berkorelasi kuat — TreeSHAP di notebook 03 akan membagi kredit di antara mereka.')
display(df.groupby('kondisi')[['tahun_pembelian','jumlah']].mean())

CEK KORELASI AWAL (umur vs harga vs depresiasi nanti)
Umur dihitung di notebook 02 sebagai REFERENCE_YEAR - tahun_pembelian.
Depresiasi adalah fungsi deterministik dari umur + harga_per_unit + kategori,
jadi ketiganya akan berkorelasi kuat — TreeSHAP di notebook 03 akan membagi kredit di antara mereka.


,tahun_pembelian,jumlah
kondisi,,
Baik,2019.15,3.78
Kurang Baik,2017.95,3.50
Rusak Berat,2017.19,1.57


## Kesimpulan 01

1. Masalah utama tetap sama: harga 55.7% kosong, kelas timpang.
2. Biner memperbaiki rasio imbalance dari 22:1 (Rusak Berat) menjadi 4.6:1 — SMOTE-NC masih perlu, tapi tidak lagi dipaksa dari 85 contoh saja.
3. Kategorikal kecil dan bersih: bahan 5 nilai, kategori 3 nilai — SMOTE-NC dengan `categorical_features=[0,1]` layak.
4. Skala numerik sangat timpang — scaler wajib sebelum SMOTE-NC.

Lanjut ke `02_preprocessing.ipynb`.